# Пара 10. LDA: темы мини-корпуса

Возьмите 6–12 документов: главы диссертации **или** набор рассказов Бунина/Чехова.  
Тема LDA ≠ мотив: имя теме даёте вы после чтения.


## Старт в Colab
1. **Файл → Сохранить копию на Диске** (чтобы правки не пропали).
2. Справа сверху — **Подключить**.
3. Ячейки сверху вниз: Play или Shift+Enter.
Базовый Colab **бесплатный**. Правило занятия: **вопрос → процедура → картинка → цитата → ограничение**.


In [ ]:
import re, requests
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import pandas as pd

BUNIN = "https://pavelaleks.github.io/samarkand_lectures/demo-corpora/bunin-temnye-allei/txt/"
files = [
    "kavkaz.txt", "holodnaya-osen.txt", "chistyj-ponedelnik.txt",
    "natali.txt", "rusya.txt", "muza.txt",
    "genrih.txt", "tanya.txt", "v-parizhe.txt",
]
# Для диссертации: docs = [Path(f).read_text(encoding="utf-8") for f in ...]
docs, titles = [], []
for fn in files:
    r = requests.get(BUNIN + fn, timeout=60); r.encoding = "utf-8"
    docs.append(r.text); titles.append(fn.replace(".txt", ""))

STOP = ["это", "как", "так", "все", "она", "он", "они", "было", "была", "были", "только", "меня", "мне", "него", "нее"]
vec = CountVectorizer(max_df=0.85, min_df=2, stop_words=STOP, token_pattern=r"(?u)\b[а-яё]{3,}\b")
X = vec.fit_transform([d.lower() for d in docs])
K = 4  # попробуйте 3, 4, 5
lda = LatentDirichletAllocation(n_components=K, random_state=42, learning_method="batch")
W = lda.fit_transform(X)
terms = vec.get_feature_names_out()

for k, topic in enumerate(lda.components_):
    top = topic.argsort()[-12:][::-1]
    print(f"Тема {k+1}:", ", ".join(terms[i] for i in top))

pd.DataFrame(W, index=titles, columns=[f"t{i+1}" for i in range(K)]).round(3)


## Сдать
Таблица: тема → ваши 3–5 слов-ярлыков → документ, где она сильна → цитата → сомнение.  
Без столбца сомнений тема в диссертацию не переносится.
